In [28]:
import arcpy
from arcpy.sa import *

# Check out the Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Define your input rasters (use exact names from your map/layers)
ndvi_current = Raster("VCIModis_NDVI_Currrent")  # 
ndvi_min = Raster("VCIModis_Minimum.crf")
ndvi_max = Raster("VCIModis_Maximum.crf")

# Compute VCI: ((NDVI_current - NDVI_min) / (NDVI_max - NDVI_min)) * 100
vci = ((ndvi_current - ndvi_min) / (ndvi_max - ndvi_min)) * 100

vci_clean = Con((vci >= 0) & (vci<= 100), vci)
del vci



In [29]:
vci_classified = Con(
    (vci_clean >= 0) & (vci_clean <= 10), 1,  # Extreme vegetation stress
    Con(
        (vci_clean > 10) & (vci_clean <= 30), 2,  # Poor condition
        Con(
            (vci_clean > 30) & (vci_clean <= 50), 3,  # Fair
            Con(
                (vci_clean > 50) & (vci_clean <= 70), 4,  # Good
                Con((vci_clean > 70) & (vci_clean <= 100), 5)  # Excellent
            )
        )
    )
)

In [39]:
#arcpy.management.BuildRasterAttributeTable(vci_classified, "Overwrite")
gdb_path = r"D:\drought\VCI\VCI.gdb"
output_name = "VCI_MODIS_11_2025_Classified"
output_path=r"D:\drought\VCI\VCI.gdb\VCI_MODIS_11_2025_Classified"
vci_classified.save(output_path)

<class 'RuntimeError'>: Unspecified error 

In [40]:
arcpy.management.AddField(output_path, "class", "TEXT")

class_labels = {
    1: "Extreme stress (Severe drought)",
    2: "Poor condition (Moderate drought)",
    3: "Fair (Normal)",
    4: "Good (Healthy)",
    5: "Excellent (Lush vegetation)"
}

<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000732: Input Table: Dataset D:\drought\VCI\VCI.gdb\VCI_MODIS_11_2025_Classified does not exist or is not supported
Failed to execute (AddField).


In [37]:
with arcpy.da.UpdateCursor(vci_classified, ["Value", "Class"]) as cursor:
    for row in cursor:
        row[1] = class_labels.get(row[0], "Unknown")
        cursor.updateRow(row)

<class 'RuntimeError'>: 'in_table' is not a table or a featureclass